# In-Class Exercise: CNN Graph Optimization (Fusion + Bandwidth)
**Time: 20 minutes**

## Learning Goals:
- Recognize common fusion opportunities in CNN computation graphs
- Estimate memory bandwidth reduction from eliminating intermediate tensor writes
- Understand the difference between inference-time and training-time optimization constraints
- Reason about data layout implications for fusion

## Instructions:
- Work through each "Your Turn" section before running the provided checks
- Think through the problems before looking at any hints or solutions
- Run cells top-to-bottom after completing each section

## Scenario: CNN Layer Sequence
You have the following CNN block running in **inference mode** (BatchNorm uses fixed statistics):

```python
def cnn_block(input_tensor):
    # Layer 1
    conv1 = conv2d(input_tensor, weights1, bias1)
    norm1 = batch_norm(conv1, scale1, offset1)
    act1 = relu(norm1)
    
    # Layer 2  
    conv2 = conv2d(act1, weights2, bias2)
    norm2 = batch_norm(conv2, scale2, offset2)
    act2 = relu(norm2)
    
    # Element-wise operations
    scaled = act2 * 0.5
    shifted = scaled + 0.1
    final = relu(shifted)
    
    return final
```

Your task is to optimize this computation graph by identifying fusion opportunities and estimating the memory bandwidth savings.

In [26]:
# Helper code - run this cell first
from dataclasses import dataclass
from typing import List

@dataclass
class Op:
    name: str
    reads: List[str] 
    writes: List[str]
    kind: str  # 'conv', 'bn', 'relu', 'eltwise', 'fused', etc.

def count_intermediate_writes(ops: List[Op], inputs: List[str], outputs: List[str]) -> int:
    """Count writes to intermediate tensors (not inputs or outputs)."""
    input_set = set(inputs)
    output_set = set(outputs)
    writes = 0
    for op in ops:
        for w in op.writes:
            if w not in input_set and w not in output_set:
                writes += 1
    return writes

def pretty_plan(ops: List[Op]) -> str:
    lines = []
    for i, op in enumerate(ops, 1):
        lines.append(f"{i:02d}. {op.kind.upper():12s} {op.name:20s} -> writes {', '.join(op.writes)}")
    return "\n".join(lines)

def summarize(ops: List[Op], inputs: List[str], outputs: List[str]) -> None:
    print(pretty_plan(ops))
    print(f"\nIntermediate writes: {count_intermediate_writes(ops, inputs, outputs)}")

In [27]:
# The current unoptimized computation graph
inputs = ["input"]
outputs = ["final"]

baseline_ops = [
    Op("conv1", ["input", "weights1", "bias1"], ["conv1"], "conv"),
    Op("bn1", ["conv1", "scale1", "offset1"], ["norm1"], "bn"), 
    Op("relu1", ["norm1"], ["act1"], "relu"),
    Op("conv2", ["act1", "weights2", "bias2"], ["conv2"], "conv"),
    Op("bn2", ["conv2", "scale2", "offset2"], ["norm2"], "bn"),
    Op("relu2", ["norm2"], ["act2"], "relu"),
    Op("scale", ["act2"], ["scaled"], "eltwise"),
    Op("shift", ["scaled"], ["shifted"], "eltwise"), 
    Op("relu3", ["shifted"], ["final"], "relu"),
]

print("Baseline (unoptimized) plan:")
summarize(baseline_ops, inputs, outputs)

Baseline (unoptimized) plan:
01. CONV         conv1                -> writes conv1
02. BN           bn1                  -> writes norm1
03. RELU         relu1                -> writes act1
04. CONV         conv2                -> writes conv2
05. BN           bn2                  -> writes norm2
06. RELU         relu2                -> writes act2
07. ELTWISE      scale                -> writes scaled
08. ELTWISE      shift                -> writes shifted
09. RELU         relu3                -> writes final

Intermediate writes: 8


## Your Turn 1: Analyze Fusion Opportunities

Look at the baseline computation graph above. For **inference mode**, analyze each potential fusion:

1. **Conv + BatchNorm**: Can these be fused? Why or why not?

2. **Conv + BatchNorm + ReLU**: Can all three be fused together? Why or why not?

3. **Element-wise chain (scale + shift + relu)**: Can these be fused? Why or why not?

4. **Cross-layer fusions**: Are there any other fusion opportunities you can identify?

**Write your analysis below, then run the next cell to check your reasoning:**

In [ ]:
# Your Turn 1 - Write your analysis here:

# 1. Conv + BatchNorm fusion:
conv_bn_fusable = True  # True or False
conv_bn_reason = "In inference mode, the BatchNorm statistics (mean and variance) and parameters (scale and offset) are fixed. Because they are constant, they can be mathematically 'folded' into the weights and biases of the preceding Convolution layer ahead of time. This means the BN layer is completely removed from the graph during runtime."

# 2. Conv + BatchNorm + ReLU fusion:  
conv_bn_relu_fusable = True  # True or False
conv_bn_relu_reason = "ReLU is a simple element-wise operation (setting negative values to zero). It can be directly fused with the Conv+BN computation. By applying the ReLU function immediately to the output values while they are still in the hardware's fast registers (SRAM), it adds virtually no computational or memory overhead."

# 3. Element-wise chain fusion:
eltwise_chain_fusable = True  # True or False  
eltwise_chain_reason = "Sequential element-wise operations can be grouped together and compiled into a single unified computation kernel. Instead of launching three separate kernels and writing intermediate results to memory, the system can compute ReLU(x * 0.5 + 0.1) in a single pass."

# 4. Other fusion opportunities:
other_fusions = ""

print("Your fusion analysis:")
print(f"Conv+BN: {conv_bn_fusable} - {conv_bn_reason}")
print(f"Conv+BN+ReLU: {conv_bn_relu_fusable} - {conv_bn_relu_reason}")  
print(f"Eltwise chain: {eltwise_chain_fusable} - {eltwise_chain_reason}")
print(f"Other opportunities: {other_fusions}")

Your fusion analysis:
Conv+BN: None - 
Conv+BN+ReLU: None - 
Eltwise chain: None - 
Other opportunities: 


In [29]:
# Check your analysis (run after completing Your Turn 1)
def check_fusion_analysis():
    correct_answers = {
        'conv_bn': True,
        'conv_bn_relu': True, 
        'eltwise_chain': True
    }
    
    explanations = {
        'conv_bn': "In inference, BN parameters are fixed, so BN can be folded into Conv weights/bias",
        'conv_bn_relu': "ReLU is element-wise and can be fused with the Conv+BN computation",
        'eltwise_chain': "Sequential element-wise ops (scale, shift, ReLU) can be combined into a single kernel"
    }
    
    print("Correct analysis:")
    for key, value in correct_answers.items():
        print(f"{key}: {value} - {explanations[key]}")
    
    print(f"\nYour answers: Conv+BN={conv_bn_fusable}, Conv+BN+ReLU={conv_bn_relu_fusable}, Eltwise={eltwise_chain_fusable}")

check_fusion_analysis()

Correct analysis:
conv_bn: True - In inference, BN parameters are fixed, so BN can be folded into Conv weights/bias
conv_bn_relu: True - ReLU is element-wise and can be fused with the Conv+BN computation
eltwise_chain: True - Sequential element-wise ops (scale, shift, ReLU) can be combined into a single kernel

Your answers: Conv+BN=None, Conv+BN+ReLU=None, Eltwise=None


## Your Turn 2: Design the Optimized Graph

Now design an optimized computation graph that takes advantage of the fusion opportunities you identified.

**Requirements:**
- Maintain the same inputs (`["input"]`) and outputs (`["final"]`)
- Use meaningful operation names for your fused operations
- Ensure the computation is mathematically equivalent to the original

In [30]:
# Your Turn 2 - Design your optimized graph here:

optimized_ops = [
    # TODO: Replace this with your optimized operation list
    # Example format:
    # Op("your_op_name", ["input_tensors"], ["output_tensors"], "op_type")
]

print("Your optimized plan:")
if optimized_ops:
    summarize(optimized_ops, inputs, outputs)
else:
    print("TODO: Complete the optimized_ops list above")

Your optimized plan:
TODO: Complete the optimized_ops list above


In [31]:
# Calculate the improvement from your optimization
if optimized_ops:
    baseline_writes = count_intermediate_writes(baseline_ops, inputs, outputs)
    opt_writes = count_intermediate_writes(optimized_ops, inputs, outputs)
    
    if baseline_writes > 0:
        reduction = (baseline_writes - opt_writes) / baseline_writes
        print(f"Baseline intermediate writes: {baseline_writes}")
        print(f"Your optimized writes: {opt_writes}")
        print(f"Reduction: {reduction:.1%}")
        
        if opt_writes <= 2:
            print("✅ Excellent optimization!")
        elif opt_writes <= 4:
            print("✅ Good optimization, but could you do better?")
        else:
            print("❌ More optimization possible - review your fusion opportunities")
    else:
        print("❌ Error in calculation")
else:
    print("Complete Your Turn 2 first")

Complete Your Turn 2 first


## Your Turn 3: Draw and Justify Your Optimized Graph

1. **Draw your optimized computation graph** using ASCII art or text description:

[Your graph here]

In [32]:
# Your Turn 3 - Complete your analysis:

# 1. ASCII graph representation:
ascii_graph = """
TODO: Draw your optimized graph here
Example format:
Input -> [Fused_Op1] -> [Fused_Op2] -> Output
"""

# 2. Fusion justifications:
fusion_justifications = """
TODO: Explain why each fusion in your graph is legal for inference
"""

# 3. Memory traffic analysis:
memory_analysis = """
TODO: Explain how your optimization reduces memory bandwidth
"""

print("Graph representation:")
print(ascii_graph)
print("\nFusion justifications:")
print(fusion_justifications)  
print("\nMemory analysis:")
print(memory_analysis)

Graph representation:

TODO: Draw your optimized graph here
Example format:
Input -> [Fused_Op1] -> [Fused_Op2] -> Output


Fusion justifications:

TODO: Explain why each fusion in your graph is legal for inference


Memory analysis:

TODO: Explain how your optimization reduces memory bandwidth



In [64]:
# Complete Reference solution (expand after attempting the exercise)

def show_solution():
    print("SOLUTION - Compare with your approach:")
    print("="*50)
    
    # YOUR TURN 1 SOLUTION (already covered in Cell 6, but for completeness):
    print("YOUR TURN 1 - Fusion Analysis:")
    print("- Conv+BN: TRUE - BN parameters fixed in inference, can fold into Conv weights")  
    print("- Conv+BN+ReLU: TRUE - ReLU is element-wise, adds no overhead when fused")
    print("- Element-wise chain: TRUE - Scale, shift, ReLU can be combined: ReLU(x*0.5+0.1)")
    
    print("\n" + "="*30)
    print("YOUR TURN 2 - Optimized Graph:")
    print("="*30)
    
    # This is what students should have created:
    solution_ops = [
        Op("fused_conv_bn_relu_1", ["input", "weights1", "bias1", "scale1", "offset1"], ["act1"], "fused"),
        Op("fused_conv_bn_relu_2", ["act1", "weights2", "bias2", "scale2", "offset2"], ["act2"], "fused"), 
        Op("fused_scale_shift_relu", ["act2"], ["final"], "fused")
    ]
    
    print("Optimal operations list:")
    for i, op in enumerate(solution_ops):
        print(f"  {i+1}. Op(\"{op.name}\", {op.reads}, {op.writes}, \"{op.kind}\")")
    
    print("\nOptimal plan:")
    summarize(solution_ops, inputs, outputs)
    
    sol_writes = count_intermediate_writes(solution_ops, inputs, outputs)
    baseline_writes = count_intermediate_writes(baseline_ops, inputs, outputs)
    reduction = (baseline_writes - sol_writes) / baseline_writes
    
    print(f"\nOptimal reduction: {reduction:.1%} (from {baseline_writes} to {sol_writes} intermediate writes)")
    
    print("\n" + "="*30)
    print("YOUR TURN 3 - Analysis:")
    print("="*30)
    
    print("\n1. ASCII Graph:")
    solution_ascii = """
Original (Baseline):
Input -> Conv1 -> BN1 -> ReLU1 -> Conv2 -> BN2 -> ReLU2 -> Scale -> Shift -> ReLU3 -> Final
         [write][write][write] [write][write][write] [write] [write]

Optimized (Fused):  
Input -> [Conv1+BN1+ReLU1] -> [Conv2+BN2+ReLU2] -> [Scale+Shift+ReLU3] -> Final
                 [write]              [write]
"""
    print(solution_ascii)
    
    print("\n2. Fusion Justifications:")
    justifications = """
- Conv+BN fusion: In inference, BN statistics are fixed, enabling mathematical folding into Conv weights
- Conv+BN+ReLU fusion: ReLU is element-wise and adds no computational or memory overhead when fused  
- Element-wise chain fusion: Scale, shift, and ReLU can be combined as: ReLU(x * 0.5 + 0.1)
"""
    print(justifications)
    
    print("\n3. Memory Traffic Analysis:")
    memory_analysis = """
- Baseline: 8 intermediate writes (every operation except final output)
- Optimized: 2 intermediate writes (only between major fused blocks)  
- Bandwidth reduction: 75% fewer memory writes
- Additional benefits: Better cache locality, fewer kernel launches, reduced memory fragmentation
"""
    print(memory_analysis)

# Uncomment to see complete solution:
show_solution()

SOLUTION - Compare with your approach:
YOUR TURN 1 - Fusion Analysis:
- Conv+BN: TRUE - BN parameters fixed in inference, can fold into Conv weights
- Conv+BN+ReLU: TRUE - ReLU is element-wise, adds no overhead when fused
- Element-wise chain: TRUE - Scale, shift, ReLU can be combined: ReLU(x*0.5+0.1)

YOUR TURN 2 - Optimized Graph:
Optimal operations list:
  1. Op("fused_conv_bn_relu_1", ['input', 'weights1', 'bias1', 'scale1', 'offset1'], ['act1'], "fused")
  2. Op("fused_conv_bn_relu_2", ['act1', 'weights2', 'bias2', 'scale2', 'offset2'], ['act2'], "fused")
  3. Op("fused_scale_shift_relu", ['act2'], ['final'], "fused")

Optimal plan:
01. FUSED        fused_conv_bn_relu_1 -> writes act1
02. FUSED        fused_conv_bn_relu_2 -> writes act2
03. FUSED        fused_scale_shift_relu -> writes final

Intermediate writes: 2

Optimal reduction: 75.0% (from 8 to 2 intermediate writes)

YOUR TURN 3 - Analysis:

1. ASCII Graph:

Original (Baseline):
Input -> Conv1 -> BN1 -> ReLU1 -> Conv2 -> B

## Extension Questions

Think about these advanced scenarios:

1. **Training vs Inference**: How would your fusion strategy change if this were training mode instead of inference? Why?

2. **Mixed Precision**: If using FP16 activations but FP32 weights, how might this affect your fusion decisions?

3. **Hardware Constraints**: What factors might make you avoid fusing everything possible?

4. **Quantization**: How would INT8 quantization change your approach to fusion?

**Discuss with your classmates or write notes below.**